# WESTPA Python API tutorial

This tutorial covers basic use of the new Python API included in WESTPA 2026.0.0a0. For the sake of familiarity, it is adapted from the [basic WESTPA tutorial (Na+ Cl- association)](https://github.com/westpa/westpa_tutorials/tree/0db92b6a2a0493f120a0b6bc00a0fdba9769bf09/tutorial7.1-basic-nacl), which uses OpenMM to run molecular dynamics and MDTraj to compute progress coordinates.

In [ ]:
import westpa

To start, we'll need two input files:
1. `bstate.pdb` &mdash; PDB file specifying the molecular topology
2. `bstate.xml` &mdash; OpenMM XML file specifying the source (initial) state of the system

Ouput files generated by previous runs can be cleaned up by running the following cell:

In [ ]:
import os
import shutil

if os.path.exists('west.h5'):
    os.remove('west.h5')
if os.path.exists('traj_segs'):
    shutil.rmtree('traj_segs')

The API components use Python's `logging` module for event logging. We'll direct that output to stdout:

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

We define the propagator using the new `OpenMMPropagator` type, which interfaces directly with the OpenMM Python API. 

In [ ]:
import openmm.app
from openmm import unit

forcefield = openmm.app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')
topology = openmm.app.PDBFile('bstate.pdb').getTopology()
system = forcefield.createSystem(
    topology,
    nonbondedMethod=openmm.app.PME,
    nonbondedCutoff=1 * unit.nanometer,
    constraints=openmm.app.HBonds,
)
system.addForce(openmm.MonteCarloBarostat(1 * unit.bar, 300 * unit.kelvin))
integrator = openmm.LangevinMiddleIntegrator(
    300 * unit.kelvin, 1 / unit.picosecond, 2 * unit.femtosecond
)

propagator = westpa.OpenMMPropagator(
    topology=topology,
    system=system,
    integrator=integrator,
    steps=1000,
    reports=[
        westpa.OpenMMReport(
            reporter_type=openmm.app.XTCReporter,
            filename='traj.xtc',
            report_interval=500,
        ),
        westpa.OpenMMReport(
            reporter_type=openmm.app.StateDataReporter,
            
            filename='log.csv',
            report_interval=100,
            options=dict(step=True, potentialEnergy=True, kineticEnergy=True, temperature=True),
        ),
    ],
)

In the cell above, note that the *reports* argument is a list of `OpenMMReport` objects, rather than OpenMM reports. The wrapper is needed because OpenMM reporters don't expose their parameters (e.g., `file`, `reportInterval`) as attributes, and we need these parameters to initialize fresh reporters each time the propagator is called.

The choice of propagator determines how microstates of the system are specified. In the case of `OpenMMPropagator`, microstates are specified by reference to XML files containing serialized OpenMM State objects, for instance, `westpa.State(ref='/path/to/state.xml')`.

To specify the progress coordinate, we define a function that takes a `Segment` object, sets its `pcoord` attribute, and returns the modified segment. Here we only compute the progress coordinate for the final configuration (i.e., `pcoord_len == 1`), since only this point is needed for resampling.

In [ ]:
import mdtraj

top = mdtraj.Topology.from_openmm(topology)

def calculate_pcoord(segment):
    traj = mdtraj.load_xml(segment.final_state.ref, top=top)
    distances = mdtraj.compute_distances(traj, atom_pairs=[[0, 1]])
    segment.pcoord = distances * 10  # nanometer -> angstrom
    return segment

We define the resampler using the new `HuberKimResampler` type, which takes a bin mapper and target counts as parameters:

In [ ]:
import numpy as np

resampler = westpa.HuberKimResampler(
    bin_mapper=westpa.RectilinearBinMapper(
        boundaries=[
            [0, 2.6, 2.8, 3, 3.2, 3.4, 3.6, 3.8, 4, 4.5, 5, 5.5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, np.inf]
        ],
    ),
    bin_target_counts=5,
)

To specify the source and sink for recycling, we use the new `Source` and `Sink` types:

In [ ]:
import os.path

initial_state = westpa.State(ref=os.path.abspath('bstate.xml'))

source = westpa.Source(states=[initial_state])
sink = westpa.Sink(indicator=lambda segment: segment.pcoord[-1, 0] < 2.6)

Finally, we can create, initialize, and run the simulation:

In [ ]:
simulation = westpa.Simulation(
    datafile='west.h5',
    resampler=resampler,
    propagator=propagator,
    pcoord_calculator=calculate_pcoord,
    source=source,
    sink=sink,
)

In [ ]:
simulation.initialize(initial_states=[initial_state] * 5)

In [ ]:
simulation.run(n_iters=10)